# Phase B0 — guarded recovery runner CUDA validation

This notebook is **dummy-only and unauthorized**. It validates the modified CUDA training, diagnostics, checkpoint, and lifecycle code. It cannot run the scientific recovery.


## 1. Confirm the frozen Tesla T4 runtime


In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
if gpu_name != 'Tesla T4':
    raise RuntimeError(f'Frozen recovery validation requires Tesla T4, found {gpu_name!r}.')


## 2. Upload exactly one recovery-validation bundle


In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.tar.gz')]
if len(archives) != 1:
    raise RuntimeError('Upload exactly one .tar.gz recovery-validation bundle.')
ARCHIVE = archives[0]
print('Uploaded:', ARCHIVE)


## 3. Restore the exact source commit and six frozen resources


In [ ]:
import json, pathlib, shutil, subprocess, sys, tarfile
extract_root = pathlib.Path('/content/phase_b0_recovery_validation_bundle')
repo_root = pathlib.Path('/content/latent-stroke-dynamics')
if extract_root.exists() or repo_root.exists():
    raise RuntimeError('Validation extraction already exists; use a fresh runtime.')
extract_root.mkdir()
with tarfile.open(ARCHIVE, 'r:gz') as archive:
    archive.extractall(extract_root)
manifest = json.loads((extract_root / 'bundle_manifest.json').read_text())
if manifest['status'] != 'phase_b0_colab_recovery_validation_bundle_unauthorized':
    raise RuntimeError('Unexpected validation bundle status.')
if manifest['recovery_authorized'] is not False or manifest['scientific_training_allowed'] is not False:
    raise RuntimeError('Bundle crossed the unauthorized dummy-only boundary.')
subprocess.run(['git', 'clone', '--branch', manifest['branch'], str(extract_root / 'repository.bundle'), str(repo_root)], check=True)
head = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip()
if head != manifest['source_commit']:
    raise RuntimeError('Restored Git commit does not match bundle manifest.')
for source in (extract_root / 'resources').rglob('*'):
    if source.is_file():
        destination = repo_root / source.relative_to(extract_root / 'resources')
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
print(json.dumps(manifest, indent=2))


## 4. Install and run the complete locked suite


In [ ]:
%cd /content/latent-stroke-dynamics
%pip install -q -e ".[dev]"


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


Expected: **138 passed**. Any failure stops the notebook.


## 5. Run dummy-only CUDA recovery implementation validation


In [ ]:
report_path = pathlib.Path('/content/phase-b0-colab-recovery-validation-report.json')
subprocess.run([sys.executable, 'experiments/24_phase_b_colab_recovery_validation.py', '--report', str(report_path)], check=True)


## 6. Inspect and download the validation report


In [ ]:
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2, sort_keys=True))
if report['status'] != 'phase_b0_colab_recovery_implementation_valid_unauthorized':
    raise RuntimeError('Recovery implementation validation did not pass.')
if report['recovery_authorized'] is not False or report['scientific_models_trained'] is not False:
    raise RuntimeError('Validation report crossed its boundary.')
files.download(str(report_path))


Send the downloaded report and test result for review. This notebook produces no scientific evidence and grants no authorization.
